# Session 17 · Gradient descent, step by step
**Data Science and Artificial Intelligence** · Generative AI and Agentic AI  
**Learner practical**

This notebook uses **four fictional, synthetic, deliberately noiseless delivery orders**. It is a mathematical exercise, not an operational ETA predictor. We will read two input features as vectors, combine them into predictions, measure error, and change **one shared distance weight** to reduce training loss.

**What you will produce:** the three short guided checks during teaching, then your own prediction/loss check, one gradient update, an iteration history, two plots driven by that history, and a saved explanation. The trainer solution is a separate file; this notebook has no file dependencies.

**Start:** Open this file in local Jupyter or Google Colab. Use a Python kernel with NumPy and Matplotlib available. Run cells in order with Shift+Enter. No upload, login, CSV, network call, package install, or extra Python module is needed for the exercise. The independent TODO cells are deliberately incomplete, so **Run All** will stop there until you fill them. If imports fail during class, use the paper-first checks and ask the trainer for the prepared environment.

## 1 · From an order to numerical inputs

A team proposes a simple rule: base handling minutes, plus minutes for each kilometre, plus minutes for each item. A **feature** is an input about one order. A **target** is the observed time we compare with the prediction.

| Case | `distance_km` | `items` | `observed_minutes` (target) |
|---:|---:|---:|---:|
| 1 | 1 | 1 | 15 |
| 2 | 2 | 1 | 18 |
| 3 | 3 | 2 | 23 |
| 4 | 4 | 2 | 26 |

The ordered pair for case 3 is **`[3, 2]`**: 3 km and 2 items. It is a two-component feature vector, or the point `(3, 2)` in a two-feature space. The 23 observed minutes are **not** a third feature. `X` stacks one order per row; its columns stay in the order `[distance_km, items]`. `y` keeps the four targets separately.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[1, 1], [2, 1], [3, 2], [4, 2]], dtype=float)
y = np.array([15, 18, 23, 26], dtype=float)
w = np.array([4, 2], dtype=float)
b = 10.0

print("X shape (orders, features):", X.shape)
print("y shape (one target per order):", y.shape)
print("w shape (one weight per feature):", w.shape)
print("Case 3 feature vector:", X[2], "→ observed", y[2], "minutes")
print("Feature columns: 0 = distance_km; 1 = items")

The rule starts as `predicted_minutes = b + a × distance_km + c × items`, where `b = 10` base **minutes**, `a = 4` **minutes per km**, and `c = 2` **minutes per item**. The inputs differ by order; `a`, `c`, and `b` are shared rule parameters. Changing a case's distance does not change the shared distance weight. Here `w = [a, c]`.

The next chart shows *where orders sit in feature space*. Its axes are distance and items. Later, the loss curve will have completely different axes: **distance weight** and **mean squared error**.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.scatter(X[:, 0], X[:, 1], s=75, color="#1767a3")
for case_number, (distance, items) in enumerate(X, start=1):
    ax.annotate(f"Case {case_number}", (distance, items), xytext=(7, 7),
                textcoords="offset points", fontsize=9)
ax.annotate("(3 km, 2 items)", xy=(3, 2), xytext=(2.5, 1.5),
            arrowprops={"arrowstyle": "->", "color": "#a33b32"}, color="#a33b32")
ax.set(xlabel="Distance (km)", ylabel="Items (count)",
       title="Four synthetic orders in two-feature space",
       xlim=(0, 4.7), ylim=(0, 2.7))
ax.set_xticks(range(5))
ax.set_yticks(range(3))
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

### Matrix operations: same numbers, different meanings

`X.shape == (4, 2)` means four rows (orders) and two columns (features). `2 * X` scales every entry; `X + X` adds matching entries. With `w.shape == (2,)`, NumPy broadcasts `X * w` across the four rows, leaving a **`(4, 2)` array of separate contributions**. `X @ w` *adds* those two contributions per row and gives four weighted sums, shape **`(4,)`**. **Before running the check below:** would `(4, 2) @ (3,)` work? Name the inner dimensions that must agree.

A compact matrix-to-matrix example uses `W = [[4, 5], [2, 2]]`. Each column of `X @ W` is a different weighted-sum column, so the result has shape `(4, 2)`. This second column is only an illustration of a different rule. A transpose, `X.T`, swaps row and column orientation to shape `(2, 4)`; we do not need it for the practical.

In [ ]:
W = np.array([[4, 5], [2, 2]], dtype=float)
print("2 * X (first row):", (2 * X)[0])
print("X + X (first row):", (X + X)[0])
print("X * w (first row and shape):", (X * w)[0], (X * w).shape)
print("X @ w (values and shape):", X @ w, (X @ w).shape)
print("X @ W (two weighted-sum columns):")
print(X @ W)
print("X @ W shape:", (X @ W).shape, "; X.T shape:", X.T.shape)
try:
    X @ np.array([4, 2, 1], dtype=float)
except ValueError:
    print("(4, 2) @ (3,) fails: 2 feature columns cannot match 3 weights.")

## 2 · Guided check 1: a dot product makes predictions
**During teaching, minutes 55–76 · about 6 minutes. Do the hand step before running the cell.**

For case 3, multiply each input by its matching weight: `[3, 2] @ [4, 2]`. What is the weighted sum? Then add the 10-minute base time. Before running, predict the shape of `X @ w + b`: why should it contain four values rather than eight? Compare your result with the prepared NumPy check. If your device is slow, do the arithmetic on paper and compare with the trainer display.

In [ ]:
case_3_weighted_sum = X[2] @ w
guided_predictions = X @ w + b
print("Case 3 dot product:", case_3_weighted_sum)
print("Case 3 prediction after base time:", guided_predictions[2], "minutes")
print("Predictions for cases 1–4:", guided_predictions, "minutes")
print("Output shape:", guided_predictions.shape)

**Read the output:** one dot product is one number. `X @ w` repeats that weighted combination across four rows; adding scalar `b` adds the same base minutes to each result. A shape mismatch means feature and weight counts differ, rather than a problem that can be repaired by switching to `*`.

## 3 · Guided check 2: error and a loss score
**During teaching, minutes 88–108 · about 6–7 minutes. Fill the four-row worksheet on paper first.**

Our signed error is always **prediction − observed**. A positive error means overprediction. Squaring each error makes both directions contribute positively. Mean squared error (MSE) averages four squared errors, so its unit is **minutes²**. An individual error belongs to one order; MSE is one score for the current rule on these four training cases.

| Case | Predicted (min) | Observed (min) | Error = predicted − observed (min) | Squared error (min²) |
|---:|---:|---:|---:|---:|
| 1 | 16 | 15 | ___ | ___ |
| 2 | 20 | 18 | ___ | ___ |
| 3 | 26 | 23 | ___ | ___ |
| 4 | 30 | 26 | ___ | ___ |

Compute the mean of the last column. Why would averaging **signed** errors alone be a different and possibly misleading score? Then run the check.

In [ ]:
guided_errors = guided_predictions - y
guided_squared_errors = guided_errors ** 2
guided_mse = np.mean(guided_squared_errors)
print("Case | predicted | observed | signed error | squared error")
for i in range(len(y)):
    print(f"{i+1:>4} | {guided_predictions[i]:>9.2f} | {y[i]:>8.2f} |"
          f" {guided_errors[i]:>12.2f} | {guided_squared_errors[i]:>13.2f}")
print(f"Mean squared error: {guided_mse:.5f} minutes²")

**From a score to a function.** Hold `b = 10` and item weight `c = 2` fixed. Let only the shared distance weight `a` change. `prediction(a) = 10 + a × X[:, 0] + 2 × X[:, 1]`. The loss function `L(a) = mean((prediction(a) − y)²)` assigns one MSE to each choice of `a`. A loss curve plots **a (min/km)** horizontally against **MSE (minutes²)** vertically. This is training loss on the four constructed records, not evidence of future accuracy.

A derivative is the local change in loss when `a` moves a little. In this one-parameter exercise, the gradient is that derivative. We use `gradient(a) = 2 × mean(X[:, 0] × (prediction(a) − y))`. A **positive** gradient means a small move to larger `a` is locally uphill; the **negative** gradient points downhill. If several weights changed, a gradient would have one component for each adjustable weight; we do not calculate that vector here.

## 4 · Guided check 3: one paper-first correction
**During teaching, minutes 126–142 · about 4 minutes. Predict the direction, then calculate before running.**

At `a = 4`, the gradient is **+15**. With learning rate `α = 0.05`, use `a_next = a_current − α × gradient`. Should `a` move up or down? Calculate the next value, then inspect the prepared arithmetic and new loss. The learning rate controls the numerical step in this example; it is not a second weight being fitted. A single useful step does not guarantee that every rate or every later step improves loss.

In [ ]:
guided_a_next = 4.0 - 0.05 * 15.0
guided_new_predictions = b + guided_a_next * X[:, 0] + 2 * X[:, 1]
guided_new_loss = np.mean((guided_new_predictions - y) ** 2)
print("First updated distance weight:", guided_a_next, "minutes/km")
print("New MSE:", guided_new_loss, "minutes²")

**Read the step:** the gradient is uphill information, so subtracting a positive scaled gradient moves `a` down. Only the shared distance weight changed. The four input rows and observed targets stayed fixed. Crossing a minimum is possible without necessarily diverging; the effect of repeated steps depends on the objective and rate.

## 5 · Independent practical: your code and your explanation
**Orientation, minutes 142–147:** the guided cells above checked taught arithmetic. They are **not** the completed independent task. The next five stages are the protected **147–170** practical. Work down the notebook, filling cells labelled **TODO**. Save the notebook after writing your interpretation. There is no hidden data or solution import.

The prepared `prediction(a)` and `loss(a)` functions below evaluate the already taught rule. The later plot helpers only draw the histories **your code computes**; they do not calculate gradients, update weights, or provide rate trajectories. Run in order. A TODO cell deliberately raises `NotImplementedError` until you replace its placeholder; the message tells you which part to fill. If behind, prioritize the single correct update, one generated trajectory, and an honest saved explanation. Every plot and comparison here is training-loss behavior for this one controlled function.

### Stage 1 · Recheck shapes, prediction, and MSE (147–151; 4 min)
The arrays and the guided calculation are already given. **You do:** replace two `None` values with a matrix-vector prediction and the mean squared error expression. **Inspect:** four predicted minutes and the initial MSE. **Meaning:** this is the starting state before any parameter update. Check that a `(4, 2)` feature matrix combines with a `(2,)` weight vector and that the loss has squared-minute units.

Hint after an attempt: `@` combines each row with `w`; error is prediction **minus** `y`, then square and average.

In [ ]:
# TODO 1: replace both None values with your own expressions.
initial_prediction = None       # X, w, b → four predictions
initial_mse = None              # mean squared (prediction − observed) error

if initial_prediction is None or initial_mse is None:
    raise NotImplementedError("TODO 1: fill initial_prediction and initial_mse, then rerun.")
print("Initial predictions (minutes):", initial_prediction)
print("Initial MSE (minutes²):", initial_mse)

### Stage 2 · Measure the gradient and take one step (151–157; 6 min)
The fixed-rule helpers are given here so the exercise stays focused on **one adjustable parameter**. `prediction(a)` returns four predicted minutes; `loss(a)` returns one MSE in minutes². **You do:** complete the one-parameter gradient and the **subtractive** update for rate `0.05`. **Inspect:** the gradient sign, updated `a`, and new MSE. **Meaning:** a positive gradient at the start says increasing `a` is locally uphill.

Hint after an attempt: multiply each signed error by its matching distance, take the mean, then multiply by 2. Update the shared weight, not the four distances.

In [ ]:
def prediction(a):
    return b + a * X[:, 0] + 2 * X[:, 1]

def loss(a):
    signed_error = prediction(a) - y
    return float(np.mean(signed_error ** 2))

a0 = float(w[0])
print("Starting a:", a0, "minutes/km; starting loss:", loss(a0), "minutes²")

In [ ]:
def gradient(a):
    signed_error = prediction(a) - y
    # TODO 2a: return 2 × mean(distance × signed_error).
    raise NotImplementedError("TODO 2a: replace this line with the gradient expression.")

learning_rate = 0.05
g0 = gradient(a0)
a1 = None  # TODO 2b: subtract learning_rate times g0 from a0.
if a1 is None:
    raise NotImplementedError("TODO 2b: fill the subtractive update for a1.")
new_loss = loss(a1)
print("Gradient at start:", g0)
print("Updated a (minutes/km):", a1)
print("Updated MSE (minutes²):", new_loss)

### Plot helpers supplied for the remaining stages
The first helper plots `loss(a)` as a curve and connects your **ordered** `(a, loss)` history from iteration 0 onward. The second draws each of your three rate histories in its own panel. Each panel has its **own y-axis** so the diverging run cannot hide the others; the high-rate panel uses a **logarithmic MSE axis**, clearly labelled. These helpers inspect values you pass in and never generate updates themselves.

In [ ]:
def plot_trajectory(a_history, loss_history, loss_function):
    a_values = np.asarray(a_history, dtype=float)
    losses = np.asarray(loss_history, dtype=float)
    if len(a_values) != len(losses) or len(a_values) < 2:
        raise ValueError("Pass matching histories including iteration 0 and at least one update.")
    grid = np.linspace(min(1.8, np.min(a_values) - 0.2),
                       max(4.3, np.max(a_values) + 0.2), 300)
    curve = np.array([loss_function(float(value)) for value in grid])
    fig, ax = plt.subplots(figsize=(8, 4.6))
    ax.plot(grid, curve, color="#537ea0", label="Loss function L(a)")
    ax.plot(a_values, losses, "o-", color="#b5493e", linewidth=1.8,
            markersize=6, label="Your ordered updates")
    for i in sorted(set([0, 1, 2, len(a_values) - 1])):
        ax.annotate(str(i), (a_values[i], losses[i]), xytext=(6, 7),
                    textcoords="offset points", fontsize=9)
    ax.set(xlabel="Distance weight a (minutes/km)",
           ylabel="Mean squared error (minutes²)",
           title="Training loss versus distance weight: iterations 0 onward")
    ax.grid(alpha=0.2)
    ax.legend()
    fig.tight_layout()
    plt.show()


def plot_rate_comparison(histories):
    if len(histories) != 3:
        raise ValueError("Pass the three core rate histories.")
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True)
    for ax, (rate, history) in zip(axes, histories.items()):
        a_values, losses = history
        steps = np.arange(len(losses))
        ax.plot(steps, losses, "o-", linewidth=1.8, markersize=5)
        ax.set_title(f"Learning rate {rate:g}")
        ax.set_xlabel("Iteration (0 = start)")
        ax.set_ylabel("MSE (minutes²)")
        ax.set_xticks(steps)
        if rate == 0.20:
            ax.set_yscale("log")
            ax.set_ylabel("MSE (minutes²; log scale)")
        ax.grid(alpha=0.25)
    fig.suptitle("Student-computed training-loss histories · separate y-axis scales")
    fig.tight_layout()
    plt.show()

### Stage 3 · Iterate and plot your path (157–163; 6 min)
One update gave a new `a`; repeated updates show a **trajectory**. **You do:** call your gradient inside the loop, subtract a scaled gradient, and append the new `a` and corresponding loss. The prepared starting entries are **iteration 0** and must stay at `a=4`, MSE `7.5`. **Inspect:** history rows and the connected points on loss-versus-`a`. **Meaning:** the line orders your successive parameter choices; the curve shows the loss for possible `a` values.

Hint after an attempt: record the original state **before** the loop; inside the loop update `a_current`, then append that updated value and `loss(a_current)`. The horizontal axis here is `a`, not iteration.

In [ ]:
def run_descent(rate, steps=7, start_a=a0):
    a_current = float(start_a)
    a_history = [a_current]       # iteration 0
    loss_history = [loss(a_current)]
    for iteration in range(steps):
        step_gradient = None  # TODO 3a: calculate gradient at current a.
        if step_gradient is None:
            raise NotImplementedError("TODO 3a: calculate step_gradient.")
        a_current = None      # TODO 3b: subtract rate × step_gradient.
        if a_current is None:
            raise NotImplementedError("TODO 3b: update a_current by subtraction.")
        # TODO 3c: append the new a_current to a_history.
        # TODO 3d: append loss at the new a_current to loss_history.
        if len(a_history) != len(loss_history) or len(a_history) != iteration + 2:
            raise NotImplementedError("TODO 3c–d: append both updated values each iteration.")
    return np.array(a_history), np.array(loss_history)

a_history, loss_history = run_descent(0.05, steps=7, start_a=a0)
for i, (a_value, mse_value) in enumerate(zip(a_history, loss_history)):
    print(f"iteration {i}: a={a_value:.8f} min/km, MSE={mse_value:.8f} min²")
plot_trajectory(a_history, loss_history, loss)

### Stage 4 · Compare three step sizes (163–166; 3 min)
**You do:** supply the three core rates and choose the **same original `a0`** as the start for every run. Reuse your `run_descent` logic. **Inspect:** the first and final states in the printed comparison, then all plotted points in the three panels. **Meaning:** compare small, effective, and excessive steps for **this function**. The y-axis scale differs by panel; use the numbers as well as the shapes before comparing magnitudes.

Hint after an attempt: the rates are `0.001`, `0.05`, and `0.20`; resetting means passing the original `a0`, not the end of a previous run. The high-rate log axis still shows actual positive MSE values.

In [ ]:
rates = [None, None, None]  # TODO 4a: the three core rates in the stated order.
comparison_start = None    # TODO 4b: reset every run to the original a0.
if any(rate is None for rate in rates) or comparison_start is None:
    raise NotImplementedError("TODO 4a–b: supply rates and original starting a.")
histories = {}
for rate in rates:
    histories[rate] = run_descent(rate, steps=7, start_a=comparison_start)

print("rate | a at 0 | MSE at 0 | a at 1 | MSE at 1 | a at 7 | MSE at 7")
for rate, (a_values, losses) in histories.items():
    print(f"{rate:>5g} | {a_values[0]:>6.3f} | {losses[0]:>8.5f} | "
          f"{a_values[1]:>6.3f} | {losses[1]:>8.5f} | "
          f"{a_values[-1]:>8.4f} | {losses[-1]:>12.5f}")
plot_rate_comparison(histories)

### Stage 5 · Save your interpretation (166–170; 4 min)
Write **4–6 substantive sentences** in the Markdown cell immediately below, then save the notebook. Name the adjustable parameter and its unit; explain why the gradient's sign leads to this update direction; compare old and first-step MSE; describe the three-rate behavior; and state one limitation of training loss on these synthetic cases. The final explanation is part of the practical, not optional decoration.

**Save your own interpretation before comparing with the trainer's private solution.** The trainer checks and selectively reveals answers only after the attempt window.

**My interpretation — replace this prompt with my own 4–6 sentences and save:**

1. The parameter I changed and its unit are …
2. The starting gradient sign means …, so the update moves …
3. The MSE changed from … to … after one step at rate 0.05.
4. The three learning rates show …
5. One limitation of this synthetic training-loss demonstration is …